# BT5 - Value Iteration & Policy Iteration

In [1]:
!pip install gymnasium


[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import gymnasium as gym
import numpy as np
import time
from IPython import display
from gymnasium.envs.toy_text.frozen_lake import FrozenLakeEnv
from gymnasium.envs.toy_text.taxi import TaxiEnv

# Lists to store execution times and iterations
values_time = []
policy_time = []
values_iterations = []  # Store iterations for Value Iteration
policy_iterations = []  # Store iterations for Policy Iteration

In [3]:
def play(env, policy, render=False):
    state, _ = env.reset()
    total_reward = 0
    steps = 0
    done = False
    while not done:
        action = policy[state]
        next_state, reward, done, info, _ = env.step(action)
        total_reward += reward
        steps += 1
        if render:
            print(env.render())
            time.sleep(0.5)
            if not done:
                display.clear_output(wait=True)
        state = next_state

    return (total_reward, steps)

In [4]:
def play_multiple_times(env, policy, max_episodes):
    success = 0
    list_of_steps = []
    for i in range(max_episodes):
        total_reward, steps = play(env, policy)

        if total_reward > 0:
            success += 1
            list_of_steps.append(steps)

    print(f'Number of successes: {success}/{max_episodes}')
    print(f'Average number of steps: {np.mean(list_of_steps)}')

In [5]:
def policy_evaluation(env, policy, max_iters=500, gamma=0.9):
    # Initialize the values of all states to be 0
    v_values = np.zeros(env.observation_space.n)

    for i in range(max_iters):
        prev_v_values = np.copy(v_values)

        # Update the value of each state
        for state in range(env.observation_space.n):
            action = policy[state]

            # Compute the q-value of the action
            q_value = 0
            for prob, next_state, reward, done in env.P[state][action]:
                q_value += prob * (reward + gamma * prev_v_values[next_state])

            v_values[state] = q_value # update v-value

        # Check convergence
        if np.all(np.isclose(v_values, prev_v_values)):
            print(f'Converged at {i}-th iteration.')
            return v_values, i

    return v_values, max_iters

In [6]:
def value_iteration(env, max_iters=500, gamma=0.9):
    # initialize
    v_values = np.zeros(env.observation_space.n)

    for i in range(max_iters):
        prev_v_values = np.copy(v_values)

        # update the v-value for each state
        for state in range(env.observation_space.n):
            q_values = []

            # compute the q-value for each action that we can perform at the state
            for action in range(env.action_space.n):
                q_value = 0
                # loop through each possible outcome
                for prob, next_state, reward, done in env.P[state][action]:
                    q_value += prob * (reward + gamma * prev_v_values[next_state])

                q_values.append(q_value)

            # select the max q-values
            best_action = np.argmax(q_values)
            v_values[state] = q_values[best_action]

        # check convergence
        if np.all(np.isclose(v_values, prev_v_values)):
            print(f'Converged at {i}-th iteration.')
            return v_values, i

    return v_values, max_iters

In [7]:
def policy_extraction(env, v_values, gamma=0.9):
    # initialize
    policy = np.zeros(env.observation_space.n, dtype=np.int32)

    # loop through each state in the environment
    for state in range(env.observation_space.n):
        q_values = []
        # loop through each action
        for action in range(env.action_space.n):
            q_value = 0
            # loop each possible outcome
            for prob, next_state, reward, done in env.P[state][action]:
                q_value += prob * (reward + gamma * v_values[next_state])

            q_values.append(q_value)

        # select the best action
        best_action = np.argmax(q_values)
        policy[state] = best_action

    return policy

In [8]:
def policy_iteration(env, max_iters=500, gamma=0.9):
    pi_values = np.zeros(env.observation_space.n)

    for i in range(max_iters):
        prev_pi_values = np.copy(pi_values)

        # Policy Evaluation
        v_values, _ = policy_evaluation(env, pi_values)

        # Policy Improvement
        pi_values = policy_extraction(env, v_values, gamma)

        if np.all(pi_values == prev_pi_values):
            print(f'Converged at {i}-th iteration.')
            return pi_values, i  # Trả đúng số vòng lặp policy chính

    return pi_values, max_iters


## Thực nghiệm trên các Toy Games

### FrozenLake-v1

In [9]:
env1 = FrozenLakeEnv(desc=None, map_name="4x4", render_mode="ansi", is_slippery=True)


In [10]:
env1.observation_space.n

np.int64(16)

In [11]:
env1.action_space.n

np.int64(4)

In [12]:
# Value Iteration
start = time.time()
optimal_v_values, iterations = value_iteration(env1)
optimal_pi_values = policy_extraction(env1, optimal_v_values)
end = time.time()

# Test Policy
play_multiple_times(env1, optimal_pi_values, 1000)
values_time.append(end - start)
values_iterations.append(iterations)



Converged at 79-th iteration.
Number of successes: 785/1000
Average number of steps: 43.394904458598724


In [13]:
# Policy Iteration
start = time.time()
optimal_pi_values, iterations = policy_iteration(env1)
end = time.time()

# Test Policy
play_multiple_times(env1, optimal_pi_values, 1000)
policy_time.append(end - start)
policy_iterations.append(iterations)

Converged at 0-th iteration.
Converged at 23-th iteration.
Converged at 59-th iteration.
Converged at 62-th iteration.
Converged at 79-th iteration.
Converged at 80-th iteration.
Converged at 5-th iteration.
Number of successes: 785/1000
Average number of steps: 43.30573248407644


In [14]:
optimal_pi_values

array([0, 3, 0, 3, 0, 0, 0, 0, 3, 1, 0, 0, 0, 2, 1, 0], dtype=int32)

### FrozenLake8x8-v1

In [15]:
env2 = FrozenLakeEnv(desc=None, map_name="8x8", render_mode="ansi", is_slippery=True)



In [16]:
env2.observation_space.n

np.int64(64)

In [17]:
env2.action_space.n


np.int64(4)

In [18]:
# Value Iteration
start = time.time()
optimal_v_values, iterations = value_iteration(env2)
optimal_pi_values = policy_extraction(env2, optimal_v_values)
end = time.time()

# Test Policy
play_multiple_times(env2, optimal_pi_values, 1000)
values_time.append(end - start)
values_iterations.append(iterations)



Converged at 117-th iteration.
Number of successes: 724/1000
Average number of steps: 75.22099447513813


In [19]:
# Policy Iteration
start = time.time()
optimal_pi_values, iterations = policy_iteration(env2)
end = time.time()

# Test Policy
play_multiple_times(env2, optimal_pi_values, 1000)
policy_time.append(end - start)
policy_iterations.append(iterations)

Converged at 27-th iteration.
Converged at 91-th iteration.
Converged at 92-th iteration.
Converged at 86-th iteration.
Converged at 90-th iteration.
Converged at 92-th iteration.
Converged at 95-th iteration.
Converged at 100-th iteration.
Converged at 112-th iteration.
Converged at 117-th iteration.
Converged at 9-th iteration.
Number of successes: 743/1000
Average number of steps: 73.9300134589502


In [20]:
optimal_pi_values

array([3, 2, 2, 2, 2, 2, 2, 2, 3, 3, 3, 3, 2, 2, 2, 1, 3, 3, 0, 0, 2, 3,
       2, 1, 3, 3, 3, 1, 0, 0, 2, 1, 3, 3, 0, 0, 2, 1, 3, 2, 0, 0, 0, 1,
       3, 0, 0, 2, 0, 0, 1, 0, 0, 0, 0, 2, 0, 1, 0, 0, 1, 1, 1, 0],
      dtype=int32)

### Taxi-v3

In [21]:
env3 = TaxiEnv()

In [22]:
env3.observation_space.n

np.int64(500)

In [23]:
env3.action_space.n

np.int64(6)

In [24]:
# Value Iteration
start = time.time()
optimal_v_values, iterations = value_iteration(env3)
optimal_pi_values = policy_extraction(env3, optimal_v_values)
end = time.time()

# Test Policy
play_multiple_times(env3, optimal_pi_values, 1000)
values_time.append(end - start)
values_iterations.append(iterations)



Converged at 116-th iteration.
Number of successes: 1000/1000
Average number of steps: 13.117


In [25]:
# Policy Iteration
start = time.time()
optimal_pi_values, iterations = policy_iteration(env3)
end = time.time()

# Test Policy
play_multiple_times(env3, optimal_pi_values, 1000)
policy_time.append(end - start)
policy_iterations.append(iterations)

Converged at 88-th iteration.
Converged at 97-th iteration.
Converged at 100-th iteration.
Converged at 101-th iteration.
Converged at 102-th iteration.
Converged at 103-th iteration.
Converged at 106-th iteration.
Converged at 109-th iteration.
Converged at 110-th iteration.
Converged at 111-th iteration.
Converged at 112-th iteration.
Converged at 115-th iteration.
Converged at 116-th iteration.
Converged at 116-th iteration.
Converged at 116-th iteration.
Converged at 116-th iteration.
Converged at 116-th iteration.
Converged at 16-th iteration.
Number of successes: 1000/1000
Average number of steps: 13.054


In [26]:
optimal_pi_values

array([4, 4, 4, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 0, 0, 3, 3,
       3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0,
       2, 2, 2, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 2, 2,
       2, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 4, 4, 4, 4,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1,
       0, 0, 1, 1, 1, 1, 2, 2, 2, 2, 0, 0, 0, 0, 2, 2, 2, 2, 1, 2, 0, 2,
       1, 1, 1, 1, 2, 2, 2, 2, 3, 3, 3, 3, 2, 2, 2, 2, 1, 2, 3, 2, 3, 3,
       3, 3, 1, 1, 1, 1, 3, 3, 3, 3, 2, 2, 2, 2, 3, 1, 3, 2, 3, 3, 3, 3,
       1, 1, 1, 1, 3, 3, 3, 3, 0, 0, 0, 0, 3, 1, 3, 0, 3, 3, 3, 3, 1, 1,
       1, 1, 3, 3, 3, 3, 0, 0, 0, 0, 3, 1, 3, 0, 1,

## Nhận xét

In [27]:
print("Results:")
print("Value Iteration times:", values_time)
print("Policy Iteration times:", policy_time)
print("Value Iteration iterations:", values_iterations)
print("Policy Iteration iterations:", policy_iterations)
print("Average Value Iteration time:", np.mean(values_time))
print("Average Policy Iteration time:", np.mean(policy_time))
print("Average Value Iteration iterations:", np.mean(values_iterations))
print("Average Policy Iteration iterations:", np.mean(policy_iterations))
print("Ratio (Policy/Value) time:", np.mean(policy_time) / np.mean(values_time))
print("Ratio (Policy/Value) iterations:", np.mean(policy_iterations) / np.mean(values_iterations))

Results:
Value Iteration times: [0.021900415420532227, 0.0772254467010498, 0.43349170684814453]
Policy Iteration times: [0.039361000061035156, 0.11284255981445312, 0.8348801136016846]
Value Iteration iterations: [79, 117, 116]
Policy Iteration iterations: [5, 9, 16]
Average Value Iteration time: 0.17753918965657553
Average Policy Iteration time: 0.3290278911590576
Average Value Iteration iterations: 104.0
Average Policy Iteration iterations: 10.0
Ratio (Policy/Value) time: 1.8532690827051512
Ratio (Policy/Value) iterations: 0.09615384615384616


# Nhận Xét